# 02 - Asset Returns by Macro Regime

**Purpose:** measure how major asset classes behaved in each macro regime (Goldilocks, Overheating, Slowdown, Stagflation), using the regimes built in `01_data_pipeline`.

**Inputs:** `data/regimes.csv` (monthly regimes, US and Euro area)
**Assets:** US-listed ETFs, monthly total returns (dividends included)
**Period:** March 2006 – latest complete month

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

regimes = pd.read_csv("../data/regimes.csv", index_col=0, parse_dates=True)
regimes[["us_regime", "ea_regime"]].tail()

,us_regime,ea_regime
2026-05-01,Overheating,Stagflation
2026-06-01,Stagflation,Stagflation
2026-07-01,Stagflation,Overheating
2026-08-01,Overheating,Overheating
2026-09-01,NaN,Overheating


## 1. Asset data

| Ticker | Asset class |
|---|---|
| SPY | US equities |
| VGK | European equities |
| TLT | US long-term Treasuries |
| SHY | US short-term Treasuries (cash proxy) |
| LQD | US investment-grade corporate bonds |
| GLD | Gold |
| DBC | Broad commodities |

Prices are adjusted for dividends and splits, so returns are total returns. The sample starts in February 2006, when the last ETF (DBC) began trading. This excludes the 2000–2002 bear market, which is a limitation of the analysis.

In [2]:
tickers = {
    "SPY": "US equities",
    "VGK": "European equities",
    "TLT": "US long Treasuries",
    "SHY": "US short Treasuries (cash-like)",
    "LQD": "US investment-grade credit",
    "GLD": "Gold",
    "DBC": "Commodities",
}

prices = yf.download(list(tickers.keys()), start="1995-01-01", auto_adjust=True, progress=False)["Close"]

prices.apply(lambda s: s.first_valid_index())

Ticker
DBC   2006-02-06
GLD   2004-11-18
LQD   2002-07-30
SHY   2002-07-30
SPY   1995-01-03
TLT   2002-07-30
VGK   2005-03-10
dtype: datetime64[s]

## 2. Monthly returns

Daily prices are converted to month-end prices, then to monthly returns. Each return is labelled with the month it covers (e.g. `2008-10-01` is the return during October 2008), so it aligns with the regime table.

Only complete months are kept: the current month is dropped automatically, so a partial month is never treated as a full one.

In [3]:
prices = prices.loc["2006-02":]

monthly_prices = prices.resample("ME").last()
monthly_prices.index = monthly_prices.index.to_period("M").to_timestamp()

returns_usd = monthly_prices.pct_change().dropna()

current_month = pd.Timestamp.today().to_period("M").to_timestamp()
returns_usd = returns_usd[returns_usd.index < current_month]

returns_usd.tail(3)
#returns_usd.loc["2008-10"]

Ticker,DBC,GLD,LQD,SHY,SPY,TLT,VGK
Date,,,,,,,
2026-07-01,0.104651,0.008578,-0.022431,0.001563,0.000348,-0.044738,0.023153
2026-08-01,0.062818,0.099263,0.003951,0.001725,0.026799,0.007324,0.011812
2026-09-01,0.032588,-0.067529,-0.033905,-0.005463,-0.003293,-0.053829,-0.050664


## 3. Returns by regime (US regimes)

Each monthly return is matched with the US regime of the same month. Average returns are annualised (monthly mean × 12).

**Important:** this is a descriptive analysis, not a trading strategy. In real time, the regime is only known one to two months later because of publication lags. The backtest in `03_backtest` accounts for this.

In [4]:
df = returns_usd.join(regimes["us_regime"], how="inner")

print(df["us_regime"].value_counts())

avg_returns = df.groupby("us_regime").mean() * 12 * 100
avg_returns.round(1)

us_regime
Slowdown       77
Overheating    66
Stagflation    54
Goldilocks     49
Name: count, dtype: int64


,DBC,GLD,LQD,SHY,SPY,TLT,VGK
us_regime,,,,,,,
Goldilocks,3.3,12.8,9.6,2.6,22.6,10.2,25.3
Overheating,16.2,8.3,4.0,1.5,20.4,2.9,13.1
Slowdown,-5.8,17.0,8.0,2.7,13.8,1.1,13.7
Stagflation,5.4,5.1,-5.6,1.0,-11.6,2.3,-22.7


## 4. Robustness: median and hit rate

Averages can be driven by a few extreme months (e.g. October 2008, March 2020). Two additional measures check this:
- **Median:** the middle month, insensitive to extremes (shown as monthly median × 12, for comparison only).
- **Hit rate:** the share of months with a positive return.

A result is considered robust when the average, the median and the hit rate point in the same direction.

In [5]:
asset_returns = df.drop(columns="us_regime")

median_returns = asset_returns.groupby(df["us_regime"]).median() * 12 * 100
hit_rate = (asset_returns > 0).groupby(df["us_regime"]).mean() * 100

print("Median (annualised, %)")
print(median_returns.round(1))
print("\nHit rate (% of positive months)")
print(hit_rate.round(0))

Median (annualised, %)
              DBC   GLD  LQD  SHY   SPY  TLT   VGK
us_regime                                         
Goldilocks    2.7  15.5  9.1  2.4  28.3  9.3  25.6
Overheating  14.5  -1.2  5.4  0.8  24.7  6.2  17.9
Slowdown     -3.1  13.3  5.3  1.3  19.8 -4.9  30.4
Stagflation  -2.2  -6.5  0.0  1.7 -10.2 -2.9 -32.8

Hit rate (% of positive months)
              DBC   GLD   LQD   SHY   SPY   TLT   VGK
us_regime                                            
Goldilocks   53.0  59.0  65.0  69.0  71.0  57.0  71.0
Overheating  61.0  48.0  64.0  62.0  80.0  56.0  61.0
Slowdown     47.0  57.0  58.0  73.0  70.0  48.0  66.0
Stagflation  48.0  48.0  50.0  63.0  43.0  46.0  37.0


## 5. Key findings (US regimes, USD returns)

1. **Stagflation is the worst regime for equities**, especially European equities (average −23%, median −33%, positive in only 37% of months). The result is consistent month after month, not driven by a single crash.
2. **Gold and commodities are unreliable inflation hedges.** Their average return in Stagflation is positive (about +5%), but their median is negative and they rise in only about half the months: the average comes from a few large gains.
3. **"Slowdown" as measured by economic data often coincides with recovering markets.** US equities gained in 70% of Slowdown months, while long Treasuries fell in more than half. Because macro data is lagged and markets are forward-looking, the slowdown is usually already priced in by the time the data confirms it.
4. **Goldilocks is favourable for almost all assets**, especially equities.

## Limitations

- Each regime has roughly 50–80 months of data, so differences of a few percentage points between regimes are not statistically reliable.
- Returns are in US dollars. A euro-based investor would see different results because of exchange-rate moves (e.g. the strong dollar in 2022). This is addressed in Section 6.
- Regimes are matched to returns in the same month, which assumes they are known immediately. This is not realistic and is addressed in the backtest.
- US growth is measured by industrial production, which covers only part of the economy.

## 6. Euro-based returns

A euro-based investor holding US-listed ETFs is exposed to both the asset and the EUR/USD exchange rate. Prices are converted to euros using the month-end EUR/USD rate (FRED: `DEXUSEU`, US dollars per euro), and monthly returns are recalculated from the euro prices.

In [6]:
url_fx = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXUSEU"
fx = pd.read_csv(url_fx, index_col=0, parse_dates=True, na_values=".")
fx = fx["DEXUSEU"].dropna()

monthly_fx = fx.resample("ME").last()
monthly_fx.index = monthly_fx.index.to_period("M").to_timestamp()

monthly_fx.tail()

observation_date
2026-06-01    1.1417
2026-07-01    1.1519
2026-08-01    1.1617
2026-09-01    1.1345
2026-10-01    1.1259
Freq: MS, Name: DEXUSEU, dtype: float64

In [7]:
monthly_prices_eur = monthly_prices.div(monthly_fx, axis=0)

returns_eur = monthly_prices_eur.pct_change().dropna()
returns_eur = returns_eur[returns_eur.index < current_month]

returns_eur.tail(3)

Ticker,DBC,GLD,LQD,SHY,SPY,TLT,VGK
Date,,,,,,,
2026-07-01,0.094870,-0.000353,-0.031088,-0.007305,-0.008510,-0.053196,0.014093
2026-08-01,0.053852,0.089989,-0.004518,-0.006726,0.018137,-0.001174,0.003276
2026-09-01,0.057344,-0.045172,-0.010743,0.018382,0.020603,-0.031144,-0.027903


In [8]:
def regime_table(returns, regime):
    df = returns.join(regime, how="inner")
    assets = df.drop(columns=regime.name)
    return assets.groupby(df[regime.name]).mean() * 12 * 100

tables = {
    "USD, US regime": regime_table(returns_usd, regimes["us_regime"]),
    "EUR, US regime": regime_table(returns_eur, regimes["us_regime"]),
    "EUR, EA regime": regime_table(returns_eur, regimes["ea_regime"]),
}

for regime in ["Stagflation", "Slowdown"]:
    print(f"{regime} (annualised average return, %)")
    print(pd.DataFrame({name: t.loc[regime] for name, t in tables.items()}).round(1))
    print()

Stagflation (annualised average return, %)
     USD, US regime  EUR, US regime  EUR, EA regime
DBC             5.4            10.7             7.6
GLD             5.1            10.6             7.5
LQD            -5.6             0.4             0.2
SHY             1.0             7.2             6.8
SPY           -11.6            -5.9             2.7
TLT             2.3             8.8             5.1
VGK           -22.7           -17.6            -8.0

Slowdown (annualised average return, %)
     USD, US regime  EUR, US regime  EUR, EA regime
DBC            -5.8            -8.1           -11.0
GLD            17.0            15.1            12.2
LQD             8.0             6.4             9.3
SHY             2.7             1.5             1.4
SPY            13.8            11.8            12.1
TLT             1.1            -0.6             7.7
VGK            13.7            11.1             4.7



### Cross-tabulation

How often are the US and the Euro area in the same regime? Full sample, 1997–2026.

In [9]:
pd.crosstab(regimes["us_regime"], regimes["ea_regime"])

ea_regime,Goldilocks,Overheating,Slowdown,Stagflation
us_regime,,,,
Goldilocks,33,9,18,3
Overheating,18,45,15,18
Slowdown,30,10,55,10
Stagflation,7,31,10,33


In [10]:
growth_agree = (regimes["us_growth_rising"] == regimes["ea_growth_rising"]).mean() * 100
inflation_agree = (regimes["us_inflation_rising"] == regimes["ea_inflation_rising"]).mean() * 100

print(f"Growth direction agrees:    {growth_agree:.0f}% of months")
print(f"Inflation direction agrees: {inflation_agree:.0f}% of months")

Growth direction agrees:    60% of months
Inflation direction agrees: 74% of months


## 7. US vs Euro area: different regimes, different results

**Separating the effects.** Moving from USD returns with US regimes to EUR returns with Euro area regimes changes two things at once. Comparing them one at a time:
- **Currency:** in US Stagflation months, every US-listed asset gained about 5–6.5 points in euros, because the dollar tended to rise during these periods. In Slowdown months the effect was small and negative (about −1 to −2.5 points). The dollar acts as a cushion in some crises, but not reliably.
- **Regimes:** using Euro area regimes instead of US regimes changes results substantially. In Slowdown, long Treasury returns rise from −1% to +8%, while European equities fall from 11% to 5%: Euro area slowdowns hurt European shares more than US ones. In Stagflation, losses on equities are smaller under Euro area regimes (European equities −8% instead of −18%; US equities slightly positive).

**Robust across all versions:**
- **European equities are the worst asset in Stagflation** (−23% USD/US regime, −18% EUR/US regime, −8% EUR/EA regime).
- **Gold is the best or joint-best asset in Slowdown** (+17%, +15%, +12%).

**How often do the regions agree?**
- Same regime in **48%** of months (vs about 25% expected by chance).
- Inflation direction agrees in **74%** of months, growth direction in only **60%**.

Inflation is largely global (energy and commodity prices affect both regions), while growth depends on local conditions. Measured by economic sentiment, Euro area growth follows its own path even more than when measured by industrial production. A European investor using US regimes would be working from a different regime more than half the time.

In [11]:
returns_usd.to_csv("../data/returns_usd.csv")
returns_eur.to_csv("../data/returns_eur.csv")